In [1]:
import re
import pandas as pd
import datetime
import mysql.connector


In [2]:
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

In [2]:
def LerFilename(revista):
    comando = f'select * from rpis_lidas where rpi="{revista}"'
    cursor.execute(comando)
    resultado = cursor.fetchall()
    print(len(resultado))
    for row in resultado:
        print(row[0])
        data_revista = row[1]
    #df = pd.DataFrame(resultado)
    #data = df[1].astype("string")
    str_data_revista = str(data_revista)
    d=datetime.datetime.strptime(str_data_revista, "%Y-%m-%d")
    #print(d.year)
    aux = str(d.strftime("%d%m%Y"))
    filename = f'revistas/RM{revista}.xml'
    return filename, str_data_revista



In [3]:
def ConverteNumero(x):
    s = str(x)
    s = s.replace("BR","")
    s = s.replace(" ","")
    pos = s.find('-')
    return s[0:pos]

In [4]:
def ConverteData(data):
    d=datetime.datetime.strptime(data, "%d/%m/%Y")
    return f'{d.year:04}-{d.month:02}-{d.day:02}'

In [5]:
import xml.etree.ElementTree as ET

def LerRPI(filename,data_revista):
    file = open(file=filename,mode='r',encoding="utf8")
    file_content_raw=file.read()
    root = ET.fromstring(file_content_raw)
    data = root.attrib.get("data")
    print(data)
    
    processos = root.findall("processo")
    numero_processos = len(processos)
    print(f"Número de registros de processos: {numero_processos}")

    process_pattern = re.compile(r"<processo\b.*?</processo>", re.DOTALL)
    process_blocks = process_pattern.findall(file_content_raw)
    num_processos = len(process_blocks)
    print(f"Número de processos encontrados: {num_processos}")

    file.close()
    data=ConverteData(data)
    if data_revista == data:
        print("Data da RPI "+data_revista+" confere com a data gravada dentro do XML "+data) 
    else:
        print("Data da RPI "+data_revista+" não confere com a data gravada dentro do XML "+data) 
    return process_blocks

# SELECT distinct(data) FROM `arquivados_ipas` WHERE year(data)=2025 Total: 20
# SELECT distinct(data) FROM `arquivados_ipas` WHERE year(data)=2024 Total: 53
# SELECT distinct(data) FROM `arquivados_ipas` WHERE year(data)=2023 Total: 52
    
# SELECT count(*) FROM `arquivados_ipas` where year(data)=2025 or year(data)=2024;
# SELECT count(*) FROM `classes_nacional` where year(data)=2025 or year(data)=2024;
# SELECT count(*) FROM `classes_nice` where year(data)=2025 or year(data)=2024;
# SELECT count(*) FROM `classes_vienna` where year(data)=2025 or year(data)=2024;
# SELECT count(*) FROM `ipas` where 1;
# SELECT count(*) FROM `madri` where year(data)=2025 or year(data)=2024;
# SELECT count(*) FROM `publicados_ipas` where 1;
# SELECT count(*) FROM `revistas4ipas` where year(data)=2025 or year(data)=2024;

revista = 2883  # 2842 - 2025-06-24;  ir de 2608 2020-12-29 a 2244 2014-01-07 total 364 revistas
#filename, data_revista = LerFilename(revista)

filename = 'revistas/RM2883.xml'
data_revista = '2026-04-07'
print(filename)  # [2025] - 2837 a 2818; [2024] - 2817 a 2765; [2023] - 2764 a 2713; [2022] - 2712 a 2661; [2021] - 2660 a 2609
print(data_revista) # [2020] - 2608 a 2557; [2019] - 2556 a 2504; [2018] - 2503 a 2452; [2017] - 2451 a 2400;
print("File: " + filename) # [2016] - 2399 a 2348; [2015] - 2347 a 2296; [2014] - 2295 a 2244
file_content = LerRPI(filename,data_revista)
print(" Total registros: " + str(len(file_content)))
print(data_revista)

# teste se faltou alguma revista
# SELECT * FROM `rpis_lidas` WHERE year(data)=2014 and data not in (SELECT distinct(data) FROM `arquivados_ipas` WHERE 1)

# SELECT PROCESSO.USERDOC_FILE_NBR as numero, ACAO.ACTION_TYP as codigo_ipas, ACAO.JOURNAL_LINKAGE_VALID_DATE as data 
# FROM MARCAS.IP_ACTION ACAO LEFT OUTER JOIN MARCAS.IP_PROC PROCESSO ON PROCESSO.PROC_NBR = ACAO.PROC_NBR 
# AND PROCESSO.PROC_TYP = ACAO.PROC_TYP WHERE EXTRACT(YEAR FROM ACAO.JOURNAL_LINKAGE_VALID_DATE) = 2014 
# AND PROCESSO.USERDOC_FILE_NBR>0 and ACAO.ACTION_TYP = '360'


revistas/RM2883.xml
2026-04-07
File: revistas/RM2883.xml
07/04/2026
Número de registros de processos: 37066
Número de processos encontrados: 37066
Data da RPI 2026-04-07 confere com a data gravada dentro do XML 2026-04-07
 Total registros: 37066
2026-04-07


In [6]:
gravar = 0 # se  =1 grava nas tabelas locais do xampp

procurador_tag = re.compile('<procurador>(.*?)</procurador>', re.DOTALL) # <procurador>Bibbiana Bertolaccini Vasconcelos</procurador>
nome_tag = re.compile('<nome>(.*?)</nome>', re.DOTALL) 
especificacao_tag = re.compile('<especificacao>(.*?)</especificacao>', re.DOTALL) 
traducao_especificacao_tag = re.compile('<traducao-especificacao>(.*?)</traducao-especificacao>', re.DOTALL) 
status_tag = re.compile('<status>(.*?)</status>', re.DOTALL)
texto_complementar_tag = re.compile('<texto-complementar>(.*?)</texto-complementar>', re.DOTALL)
texto_sobrestamento_tag = re.compile('<texto-sobrestamento>(.*?)</texto-sobrestamento>', re.DOTALL)
apostila_tag = re.compile('<apostila>(.*?)</apostila>', re.DOTALL)
cedentes_tag = re.compile('<cedentes>(.*?)</cedentes>', re.DOTALL) 
cessionarios_tag = re.compile('<cessionarios>(.*?)</cessionarios>', re.DOTALL) 

nome_razao_social_pattern = re.compile(r'nome-razao-social="([^"]+)"', re.DOTALL)
pais_pattern = re.compile(r'pais="([^"]+)"', re.DOTALL)
uf_pattern = re.compile(r'uf="([^"]+)"', re.DOTALL)
codigo_pattern = re.compile(r'codigo="IPAS(\d+)"', re.DOTALL)
nome_pattern = re.compile(r'nome="([^"]+)"', re.DOTALL)
apresentacao_pattern = re.compile(r'apresentacao="([^"]+)"', re.DOTALL)
natureza_pattern = re.compile(r'natureza="([^"]+)"', re.DOTALL)
codigo_vienna_pattern = re.compile(r'codigo="([^"]+)"', re.DOTALL)
edicao_vienna_pattern = re.compile(r'edicao="(\d+)"', re.DOTALL)
codigo_nice_pattern = re.compile(r'codigo="(\d+)"', re.DOTALL)
numero_pattern = re.compile(r'processo numero="(\d+)"', re.DOTALL)
data_deposito_pattern = re.compile(r'data-deposito="([^"]+)"', re.DOTALL)
data_prioridade_pattern = re.compile(r'data="([^"]+)"', re.DOTALL)
numero_prioridade_pattern = re.compile(r'numero="([^"]+)"', re.DOTALL)
pais_prioridade_pattern = re.compile(r'pais="([^"]+)"', re.DOTALL)
protocolo_numero_pattern = re.compile(r'numero="([^"]+)"', re.DOTALL)
protocolo_data_pattern = re.compile(r'data="([^"]+)"', re.DOTALL)
protocolo_codigoServico_pattern = re.compile(r'codigoServico="([^"]+)"', re.DOTALL)
protocolo_nome_razao_social_pattern = re.compile(r'nome-razao-social="([^"]+)"', re.DOTALL)
protocolo_pais_pattern = re.compile(r'pais="([^"]+)"', re.DOTALL)
protocolo_uf_pattern = re.compile(r'uf="([^"]+)"', re.DOTALL)
data_concessao_pattern = re.compile(r'data-concessao="([^"]+)"', re.DOTALL)
data_vigencia_pattern = re.compile(r'data-vigencia="([^"]+)"', re.DOTALL)
codigo_classe_nacional_pattern = re.compile(r'<classe-nacional codigo="(\d+)"', re.DOTALL)
codigo_subclasse_nacional_pattern = re.compile(r'<sub-classe-nacional codigo="(\d+)"', re.DOTALL)
numero_madri_pattern = re.compile(r'numero-inscricao-internacional="([^"]+)"', re.DOTALL)
data_recebimento_inpi_pattern = re.compile(r'data-recebimento-inpi="([^"]+)"', re.DOTALL)
sobrestadores_processo_pattern = re.compile(r'processo="([^"]+)"', re.DOTALL)
sobrestadores_marca_pattern = re.compile(r'marca="([^"]+)"', re.DOTALL)

arquivo = open("comandos.sql", "w", encoding="utf-8")

for line in file_content:
    
    numero = ''
    data_deposito = None
    data_concessao = None
    data_vigencia = None
    numero_internacional = ''
    data_recebimento = None
    codigo_ipas = ''
    nome_ipas = ''
    texto_complementar = []
    texto_sobrestamento = []
    texto_protocolo = []
    protocolo_numero = []
    protocolo_data = []
    protocolo_codigo_servico = []
    protocolo_procurador = []
    protocolo_nome_razao_social = []
    protocolo_pais = []
    protocolo_uf = []
    texto_cessionario = [] 
    texto_cedente = []
    cedente_nome = []
    cedente_pais = []
    cedente_uf = []
    cessionario_nome = []
    texto_titulares = []
    texto_sobrestador = []
    marca_natureza = ''
    marca_apresentacao = ''
    marca_nome = ''
    classificacao_vienna_codigo = []
    classificacao_vienna_edicao = []
    classificacao_nice_codigo = []
    classificacao_nice_especificacao = []
    classificacao_nice_traducao = []
    classificacao_nice_status = []
    classificacao_nacional_classe = []
    classificacao_nacional_especificacao = []
    classificacao_nacional_subclasse = []
    texto_prioridade = []
    
    print('\n')
    numero_matches = numero_pattern.findall(line)
    for numero in numero_matches:
        print(f"Número: {numero}")
    data_deposito_matches = data_deposito_pattern.findall(line)
    for data_deposito in data_deposito_matches:
        print(f"Data depósito: {ConverteData(data_deposito)}")
        data_deposito = ConverteData(data_deposito)
    data_concessao_matches = data_concessao_pattern.findall(line)
    for data_concessao in data_concessao_matches:
        print(f"Data concessão: {ConverteData(data_concessao)}")
        data_concessao = ConverteData(data_concessao)
    data_vigencia_matches = data_vigencia_pattern.findall(line)
    for data_vigencia in data_vigencia_matches:
        print(f"Data vigência: {ConverteData(data_vigencia)}")
        data_vigencia = ConverteData(data_vigencia)
    
    dados_de_madris=re.findall("<dados-de-madri [\s\S]*/>",line)
    for dados_de_madri in dados_de_madris:
        numero_matches = numero_madri_pattern.findall(dados_de_madri)
        data_recebimento_inpi_matches = data_recebimento_inpi_pattern.findall(dados_de_madri)
        i = 0
        for match in numero_matches:
            print(f"Número Inscrição Internacional: {match}")
            numero_internacional = match
            if i < len(data_recebimento_inpi_matches):
                print(f"Data recebimento INPI: {data_recebimento_inpi_matches[i]}")
                data_recebimento = ConverteData(data_recebimento_inpi_matches[i])
            i = i + 1

            
    nome_razao_social_pattern = re.compile(r'nome-razao-social="([^"]+)"', re.DOTALL)
    titulares=re.findall("<titulares>[\s\S]*</titulares>",line)
    for titular in titulares:
        nome_razao_social_matches = nome_razao_social_pattern.findall(titular)
        pais_matches = pais_pattern.findall(titular)
        uf_matches = uf_pattern.findall(titular)
        i = 0
        for match in nome_razao_social_matches:
            print(f"Titular Nome/Razão Social: {match}")
            ref = match
            if i < len(pais_matches):
                print(f"País: {pais_matches[i]}")
                ref = f"{match} [{pais_matches[i]}]"
            if i < len(uf_matches):
                print(f"UF: {uf_matches[i]}")
                ref = f"{match} [{pais_matches[i]}/{uf_matches[i]}]"
            texto_titulares.append(ref)
            i = i + 1

    sobrestadores=re.findall("<sobrestadores>[\s\S]*</sobrestadores>",line)
    for sobrestador in sobrestadores:
        processo_matches = sobrestadores_processo_pattern.findall(sobrestador)
        marca_matches = sobrestadores_marca_pattern.findall(sobrestador)
        i = 0
        for match in processo_matches:
            print(f"Sobrestador processo: {match}")
            ref = match
            if i < len(marca_matches):
                print(f"Marca: {marca_matches[i]}")
                ref = f"{match}, marca: {marca_matches[i]}"
            texto_sobrestador.append(ref)
            i = i + 1

    # <marca apresentacao="Figurativa" natureza="Produtos e/ou Serviço"/>
    marcas_autocontidos = re.findall(r'<marca apresentacao="([^"]+)" natureza="([^"]+)"\s*/>', line)
    for marca_apresentacao, marca_natureza in marcas_autocontidos:
        print(f"Marca apresentacao: {marca_apresentacao}")
        print(f"Marca apresentação: {marca_natureza}")

    marcas=re.findall("<marca [\s\S]*</marca>",line)
    for marca in marcas:
        apresentacao_matches = apresentacao_pattern.findall(marca)
        natureza_matches = natureza_pattern.findall(marca)
        i = 0
        for match in apresentacao_matches:
            print(f"apresentacao: {match}")
            marca_apresentacao = match
            if i < len(natureza_matches):
                print(f"natureza: {natureza_matches[i]}")
                marca_natureza = natureza_matches[i]
            i = i + 1
        nomes=nome_tag.findall(marca)
        for nome in nomes:
            print(f"nome: {nome}")
            marca_nome = nome

    classes_vienna=re.findall("<classes-vienna>[\s\S]*</classes-vienna>",line)
    for classe_vienna in classes_vienna:
        classes_vienna2=re.findall("<classe-vienna [\s\S]*/>",classe_vienna)
        for classe_vienna1 in classes_vienna2:
            codigo_vienna_matches = codigo_vienna_pattern.findall(classe_vienna1)
            edicao_vienna_matches = edicao_vienna_pattern.findall(classe_vienna1)
            i = 0
            for match in codigo_vienna_matches:
                print(f"Codigo: {match}")
                classificacao_vienna_codigo.append(match)
                if i < len(edicao_vienna_matches):
                    print(f"edicao: {edicao_vienna_matches[i]}")
                    classificacao_vienna_edicao.append(edicao_vienna_matches[i])
                i = i + 1

    listas_classe_nice=re.findall("<lista-classe-nice>[\s\S]*</lista-classe-nice>",line)
    for lista_classe_nice in listas_classe_nice:
        classes_nice=re.findall("<classe-nice [\s\S]*</classe-nice>",lista_classe_nice)
        for classe_nice in classes_nice:
            codigo_nice_matches = codigo_nice_pattern.findall(classe_nice)
            especificacao_matches=especificacao_tag.findall(classe_nice)
            traducao_especificacao_matches=traducao_especificacao_tag.findall(classe_nice)
            status_matches=status_tag.findall(classe_nice)
            i = 0
            for match in codigo_nice_matches:
                print(f"Codigo Nice: {match}")
                classificacao_nice_codigo.append(match)
                if i < len(especificacao_matches):
                    print(f"especificação: {especificacao_matches[i]}")
                    especificacao_matches[i] = especificacao_matches[i].replace("'", "''")
                    classificacao_nice_especificacao.append(especificacao_matches[i])
                if i < len(traducao_especificacao_matches):
                    print(f"tradução especificação: {traducao_especificacao_matches[i]}")
                    traducao_especificacao_matches[i] = traducao_especificacao_matches[i].replace("'", "''")
                    classificacao_nice_traducao.append(traducao_especificacao_matches[i])
                if i < len(status_matches):
                    print(f"status: {status_matches[i]}")
                    classificacao_nice_status.append(status_matches[i])
                i = i + 1

    classe_nacionals=re.findall("<classe-nacional[\s\S]*?</classe-nacional>",line)
    i = 0
    for classe_nacional in classe_nacionals:
        #print(f"teste {classe_nacional}")
        codigo_classe_nacional_matches = codigo_classe_nacional_pattern.findall(classe_nacional)
        for match in codigo_classe_nacional_matches:
            print(f"Codigo Classe nacional: {match}")
            classificacao_nacional_classe.append(match)
        especificacao_matches = especificacao_tag.findall(classe_nacional)
        for match in especificacao_matches:
            print(f"especificacao: {match}")
            match = match.replace("'", "''")
            classificacao_nacional_especificacao.append(match)
        subclasses_nacional=re.findall("<sub-classes-nacional>[\s\S]*</sub-classes-nacional>",classe_nacional)
        for subclasse_nacional in subclasses_nacional:
            codigo_matches = codigo_subclasse_nacional_pattern.findall(subclasse_nacional)
            classificacao_nacional_subclasse.append(', '.join(codigo_matches))
            for match in codigo_matches:
                print(f"Codigo subclasse {i}: {match}")
        i = i + 1
                
    prioridades_unionista=re.findall("<prioridade-unionista>[\s\S]*?</prioridade-unionista>",line)
    for prioridade_unionista in prioridades_unionista:
        data_prioridade_matches = data_prioridade_pattern.findall(prioridade_unionista)
        numero_prioridade_matches = numero_prioridade_pattern.findall(prioridade_unionista)
        pais_prioridade_matches = pais_prioridade_pattern.findall(prioridade_unionista)
        i = 0
        for match in data_prioridade_matches:
            print(f"Data prioridade: {match}")
            data_prioridade = match
            numero_prioridade = ''
            pais_prioridade = ''
            if i < len(numero_prioridade_matches):
                print(f"Número prioridade: {numero_prioridade_matches[i]}")
                numero_prioridade = numero_prioridade_matches[i]
            if i < len(pais_prioridade_matches):
                print(f"País prioridade: {pais_prioridade_matches[i]}")
                pais_prioridade = pais_prioridade_matches[i]
            texto_prioridade.append(f"[{pais_prioridade}] {numero_prioridade} de {data_prioridade}")
            i = i + 1

    apostila = ''
    apostilas=apostila_tag.findall(line)
    for apostila in apostilas:
        print(f"apostila: {apostila}")

    procurador = ''
    procuradores=procurador_tag.findall(line)
    if procuradores:  # Verifica se a lista não está vazia
        for i, procurador in enumerate(procuradores):
            if i == len(procuradores) - 1:  # Verifica se é o último elemento
                print(f"Procurador: {procurador}")
        procurador = ', '.join(procuradores)
    else:
        print("A lista de procuradores está vazia.")
        
       
    #despachos_com_conteudo = re.findall(r'<despacho codigo="IPAS(\d+)" nome="([^"]+)">([\s\S]*?)</despacho>', line)
    despachos_com_conteudo = re.findall(r'<despacho codigo="IPAS(\d+)"(?: nome="([^"]+)")?(?:>([\s\S]*?)</despacho>|/>)', line)
    for codigo_ipas, nome_ipas, despacho in despachos_com_conteudo:
        texto_complementar = []
        texto_sobrestamento = []
        texto_protocolo = []
        protocolo_numero = []
        protocolo_data = []
        protocolo_codigo_servico = []
        protocolo_procurador = []
        protocolo_nome_razao_social = []
        protocolo_pais = []
        protocolo_uf = []
        texto_protocolo = []
        cedente_nome = []
        cedente_pais = []
        cedente_uf = []
        texto_cedente = []
        cessionario_nome = []
        texto_cessionario = []

        print(f"Despacho Código: IPAS{codigo_ipas}")
        print(f"Nome: {nome_ipas}")
        data = data_revista
        cmd = f"INSERT INTO arquivados_ipas (id, codigo_ipas, numero, data, anulado, prmexame) VALUES (NULL, '{codigo_ipas}', '{numero}', '{data}', 0, 0);"
        arquivo.write(cmd + "\n")
        print(cmd)
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()
            print("gravado")
        
        texto_complementar_matches = texto_complementar_tag.findall(despacho)
        for match in texto_complementar_matches:
            print(f"texto complementar: {match}")
            #texto_complementar = match
            texto_complementar.append(match)
        texto_sobrestamento_matches = texto_sobrestamento_tag.findall(despacho)
        for match in texto_sobrestamento_matches:
            print(f"texto sobrestamento: {match}")
            #texto_sobrestamento = match
            texto_sobrestamento.append(match)
        protocolos=re.findall("<protocolo [\s\S]*</protocolo>",despacho)
        for protocolo in protocolos:
            protocolo_numero_matches = protocolo_numero_pattern.findall(protocolo)
            protocolo_data_matches = protocolo_data_pattern.findall(protocolo)
            protocolo_codigoServico_matches = protocolo_codigoServico_pattern.findall(protocolo)
            i = 0
            for match in protocolo_numero_matches:
                print(f"Protocolo numero: {match}")
                protocolo_numero.append(match)
                if i < len(protocolo_data_matches):
                    print(f"data: {protocolo_data_matches[i]}")
                    protocolo_data.append(protocolo_data_matches[i])
                if i < len(protocolo_codigoServico_matches):
                    print(f"Código Serviço: {protocolo_codigoServico_matches[i]}")
                    protocolo_codigo_servico.append(protocolo_codigoServico_matches[i])
                i = i + 1
            procuradores=procurador_tag.findall(protocolo)
            i = 0
            for procurador in procuradores:
                print(f"procurador: {procurador}")
                protocolo_procurador.append(procurador)
            protocolo_nome_razao_social_matches=protocolo_nome_razao_social_pattern.findall(protocolo)
            protocolo_pais_matches=protocolo_pais_pattern.findall(protocolo)
            protocolo_uf_matches=protocolo_uf_pattern.findall(protocolo)
            i = 0
            for nome_razao_social in protocolo_nome_razao_social_matches:
                print(f"Nome razão social: {nome_razao_social}")
                protocolo_nome_razao_social.append(nome_razao_social)
                if i < len(protocolo_pais_matches):
                    print(f"País: {protocolo_pais_matches[i]}")
                    protocolo_pais.append(protocolo_pais_matches[i])
                if i < len(protocolo_uf_matches):
                    print(f"UF: {protocolo_uf_matches[i]}")
                    protocolo_uf.append(protocolo_uf_matches[i])
                i = i + 1
            i = 0
            for elemento in protocolo_numero:
                if i < len(protocolo_data):
                    str_data = protocolo_data[i]
                else:
                    str_data = ''
                if i < len(protocolo_codigo_servico):
                    str_codigo_servico = protocolo_codigo_servico[i]
                else:
                    str_codigo_servico = ''
                if i < len(protocolo_procurador):
                    str_procurador = protocolo_procurador[i]
                else:
                    str_procurador = ''
                if i < len(protocolo_nome_razao_social):
                    str_nome_razao_social = protocolo_nome_razao_social[i]
                else:
                    str_nome_razao_social = ''
                if i < len(protocolo_pais):
                    str_pais = protocolo_pais[i]
                else:
                    str_pais = ''
                if i < len(protocolo_uf):
                    str_uf = protocolo_uf[i]
                else:
                    str_uf = ''
                texto_protocolo.append(f"protocolo {elemento} de {str_data} e " \
                                  f"código de serviço {str_codigo_servico}, procurador: " \
                                  f"{str_procurador}, requerente: {str_nome_razao_social}" \
                                  f"[{str_pais}/{str_uf}]")
                i = i + 1
            cedentes=cedentes_tag.findall(protocolo)
            for cedente in cedentes:
                nome_razao_social_matches=protocolo_nome_razao_social_pattern.findall(cedente)
                pais_matches=protocolo_pais_pattern.findall(cedente)
                uf_matches=protocolo_uf_pattern.findall(cedente)
                i = 0
                for nome_razao_social in nome_razao_social_matches:
                    print(f"Cedente Nome razão social: {nome_razao_social}")
                    cedente_nome.append(nome_razao_social)
                    texto_nome = nome_razao_social
                    texto_pais = ''
                    texto_uf = ''
                    if i < len(pais_matches):
                        print(f"País: {pais_matches[i]}")
                        cedente_pais.append(pais_matches[i])
                        texto_pais = pais_matches[i]
                    if i < len(uf_matches):
                        print(f"UF: {uf_matches[i]}")
                        cedente_uf.append(uf_matches[i])
                        texto_uf = uf_matches[i]
                    texto_cedente.append(f"Cedente: {texto_nome} [{texto_pais}/{texto_uf}]")
                    i = i + 1
            cessionarios=cessionarios_tag.findall(protocolo)
            for cessionario in cessionarios:
                nome_razao_social_matches=protocolo_nome_razao_social_pattern.findall(cessionario)
                i = 0
                for nome_razao_social in nome_razao_social_matches:
                    print(f"Cessionário Nome razão social: {nome_razao_social}")
                    cessionario_nome.append(nome_razao_social)
                    texto_cessionario.append(f"Cessionário: {cessionario_nome[i]}")
                    i = i + 1

        if numero_internacional != '':
            cmd = f"INSERT IGNORE INTO madri (numero, numero_internacional, data, data_recebimento) VALUES ('{numero}', '{numero_internacional}', '{data}', '{data_recebimento}');"
            print(cmd)
            numero_internacional = ''
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()

        if nome_ipas != '':
            cmd = f"INSERT IGNORE INTO ipas (codigo_ipas, nome_ipas) VALUES ('{codigo_ipas}', '{nome_ipas}');"
            print(cmd)
            nome_ipas = ''
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        
        if texto_complementar:
            for i in range(len(texto_complementar)):
                texto_complementar[i] = texto_complementar[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_complementar[i]}', 'texto_complementar');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()

        if texto_sobrestamento:
            for i in range(len(texto_sobrestamento)):
                texto_sobrestamento[i] = texto_sobrestamento[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestamento[i]}', 'texto_sobrestamento');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()

        if texto_protocolo:
            for i in range(len(texto_protocolo)):
                texto_protocolo[i] = texto_protocolo[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_protocolo[i]}', 'texto_protocolo');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            texto_protocolo = []
            
        if texto_cessionario:
            for i in range(len(texto_cessionario)):
                texto_cessionario[i] = texto_cessionario[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cessionario[i]}', 'cessionarios');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            texto_cessionario =[]

        if texto_cedente:
            for i in range(len(texto_cedente)):
                texto_cedente[i] = texto_cedente[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cedente[i]}', 'cedentes');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            texto_cedente = []
        #if texto_titulares:
        #    for i in range(len(texto_titulares)):
        #        texto_titulares[i] = texto_titulares[i].replace("'", " ").rstrip('\\')
        #        cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_titulares[i]}', 'titular');"
        #        print(cmd)
        #        arquivo.write(cmd + "\n")

        if texto_sobrestador:
            for i in range(len(texto_sobrestador)):
                texto_sobrestador[i] = texto_sobrestador[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestador[i]}', 'titular');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            texto_sobrestador = []

        if codigo_ipas == '009' or codigo_ipas == '576':
            titulares=', '.join(texto_titulares)
            str_deposito = f"'{data_deposito}'"
            if data_deposito is None:
                str_deposito = "NULL"
            str_concessao = f"'{data_concessao}'"
            if data_concessao is None:
                str_concessao = "NULL"
            str_vigencia = f"'{data_vigencia}'"
            if data_vigencia is None:
                str_vigencia = "NULL"
            #titulares = titulares.replace("'", "\\'")
            marca_nome = marca_nome.replace("'", " ").rstrip('\\')
            titulares = titulares.replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO publicados_ipas (numero, titular, data_deposito, data_concessao, data_vigencia, data_out, despacho_out, apresentacao, natureza, nome) VALUES ('{numero}', '{titulares}', {str_deposito}, {str_concessao}, {str_vigencia}, NULL, '', '{marca_apresentacao}', '{marca_natureza}', '{marca_nome}');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()

        if classificacao_vienna_codigo:
            for i in range(len(classificacao_vienna_codigo)):
                cmd = f"INSERT IGNORE INTO classes_vienna (id, numero, codigo, edicao, data) VALUES (NULL, '{numero}', '{classificacao_vienna_codigo[i]}', {classificacao_vienna_edicao[i]}, '{data}');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            classificacao_vienna_codigo = []

        if classificacao_nice_codigo:
            for i in range(len(classificacao_nice_codigo)):
                if i >= len(classificacao_nice_codigo):
                    str_codigo = "NULL"
                else:
                    str_codigo = f"'{classificacao_nice_codigo[i]}'"
                if i >= len(classificacao_nice_especificacao):
                    str_especificacao = "NULL"
                else:
                    classificacao_nice_especificacao[i] = classificacao_nice_especificacao[i].replace("'", " ").rstrip('\\')
                    str_especificacao = f"'{classificacao_nice_especificacao[i]}'"
                if i >= len(classificacao_nice_traducao):
                    str_traducao = "NULL"
                else:
                    str_traducao = f"'{classificacao_nice_traducao[i]}'"
                if i >= len(classificacao_nice_status):
                    str_status = "NULL"
                else:
                    str_status = f"'{classificacao_nice_status[i]}'"
                cmd = f"INSERT IGNORE INTO classes_nice (id, numero, codigo, especificacao, traducao, status, data) VALUES (NULL, '{numero}', {str_codigo}, {str_especificacao}, {str_traducao}, {str_status}, '{data}');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            classificacao_nice_codigo = []

        if classificacao_nacional_classe:
            for i in range(len(classificacao_nacional_classe)):
                if i >= len(classificacao_nacional_classe):
                    str_classe = "NULL"
                else:
                    str_classe = f"'{classificacao_nacional_classe[i]}'"
                if i >= len(classificacao_nacional_especificacao):
                    str_especificacao = "NULL"
                else:
                    classificacao_nacional_especificacao[i] = classificacao_nacional_especificacao[i].replace("'", " ").rstrip('\\')
                    str_especificacao = f"'{classificacao_nacional_especificacao[i]}'"

                if i >= len(classificacao_nacional_subclasse):
                    str_subclasse = "NULL"
                else:
                    str_subclasse = f"'{classificacao_nacional_subclasse[i]}'"
                cmd = f"INSERT IGNORE INTO classes_nacional (id, numero, codigo_classe, especificacao, codigo_subclasse, data) VALUES (NULL, '{numero}', {str_classe}, {str_especificacao}, {str_subclasse}, '{data}');"
                print(cmd)
                arquivo.write(cmd + "\n")
                if gravar: 
                    cursor.execute(cmd)
                    conexao.commit()
            classificacao_nacional_classe = []

        if texto_prioridade:
            prioridades=', '.join(texto_prioridade)
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{prioridades}', 'prioridade');"
            print(cmd)
            texto_prioridade = ''
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()

        if apostila != '':
            apostila = apostila.replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{apostila}', 'apostila');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
            apostila = ''

        if procurador != '':
            procurador = procurador.replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{procurador}', 'procurador');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
            procurador = ''

    despachos_autocontidos = re.findall(r'<despacho codigo="IPAS(\d+)" nome="([^"]+)"\s*/>', line)
    for codigo_ipas, nome_ipas in despachos_autocontidos:
        print(f"Despacho Código: IPAS{codigo_ipas}")
        print(f"Nome: {nome_ipas}")
        data = data_revista
        cmd = f"INSERT IGNORE INTO arquivados_ipas (id, codigo_ipas, numero, data, anulado, prmexame) VALUES (NULL, '{codigo_ipas}', '{numero}', '{data}', 0, 0);"
        print(cmd)
        arquivo.write(cmd + "\n")
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

    if procurador != '':
        procurador = procurador.replace("'", " ").rstrip('\\')
        cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{procurador}', 'procurador');"
        print(cmd)
        arquivo.write(cmd + "\n")
        procurador = ''
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

    if numero_internacional != '':
        cmd = f"INSERT IGNORE INTO madri (numero, numero_internacional, data, data_recebimento) VALUES ('{numero}', '{numero_internacional}', '{data}', '{data_recebimento}');"
        print(cmd)
        numero_internacional = ''
        arquivo.write(cmd + "\n")
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

    if nome_ipas != '':
        cmd = f"INSERT IGNORE INTO ipas (codigo_ipas, nome_ipas) VALUES ('{codigo_ipas}', '{nome_ipas}');"
        print(cmd)
        nome_ipas = ''
        arquivo.write(cmd + "\n")
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

    if texto_complementar:
        for i in range(len(texto_complementar)):
            texto_complementar[i] = texto_complementar[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_complementar[i]}', 'texto_complementar');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
            
    if texto_sobrestamento:
        for i in range(len(texto_sobrestamento)):
            texto_sobrestamento[i] = texto_sobrestamento[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestamento[i]}', 'texto_sobrestamento');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()

    if texto_protocolo:
        for i in range(len(texto_protocolo)):
            texto_protocolo[i] = texto_protocolo[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_protocolo[i]}', 'texto_protocolo');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        texto_protocolo = []

    if texto_cessionario:
        for i in range(len(texto_cessionario)):
            texto_cessionario[i] = texto_cessionario[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cessionario[i]}', 'cessionarios');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        texto_cessionario =[]

    if texto_cedente:
        for i in range(len(texto_cedente)):
            texto_cedente[i] = texto_cedente[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cedente[i]}', 'cedentes');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        texto_cedente = []

    if texto_titulares:
        for i in range(len(texto_titulares)):
            texto_titulares[i] = texto_titulares[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_titulares[i]}', 'titular');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()

    if texto_sobrestador:
        for i in range(len(texto_sobrestador)):
            texto_sobrestador[i] = texto_sobrestador[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestador[i]}', 'titular');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        texto_sobrestador = []

    if codigo_ipas == '009' or codigo_ipas == '576':
        titulares=', '.join(texto_titulares)
        str_deposito = f"'{data_deposito}'"
        if data_deposito is None:
            str_deposito = "NULL"
        str_concessao = f"'{data_concessao}'"
        if data_concessao is None:
            str_concessao = "NULL"
        str_vigencia = f"'{data_vigencia}'"
        if data_vigencia is None:
            str_vigencia = "NULL"
        #titulares = titulares.replace("'", "\\'")
        marca_nome = marca_nome.replace("'", " ").rstrip('\\')
        cmd = f"INSERT IGNORE INTO publicados_ipas (numero, titular, data_deposito, data_concessao, data_vigencia, data_out, despacho_out, apresentacao, natureza, nome) VALUES ('{numero}', '{titulares}', {str_deposito}, {str_concessao}, {str_vigencia}, NULL, '', '{marca_apresentacao}', '{marca_natureza}', '{marca_nome}');"
        print(cmd)
        arquivo.write(cmd + "\n")
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

    if classificacao_vienna_codigo:
        for i in range(len(classificacao_vienna_codigo)):
            cmd = f"INSERT IGNORE INTO classes_vienna (id, numero, codigo, edicao, data) VALUES (NULL, '{numero}', '{classificacao_vienna_codigo[i]}', {classificacao_vienna_edicao[i]}, '{data}');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        classificacao_vienna_codigo = []

    if classificacao_nice_codigo:
        for i in range(len(classificacao_nice_codigo)):
            if i >= len(classificacao_nice_codigo):
                str_codigo = "NULL"
            else:
                str_codigo = f"'{classificacao_nice_codigo[i]}'"
            if i >= len(classificacao_nice_especificacao):
                str_especificacao = "NULL"
            else:
                classificacao_nice_especificacao[i] = classificacao_nice_especificacao[i].replace("'", " ").rstrip('\\')
                str_especificacao = f"'{classificacao_nice_especificacao[i]}'"
            if i >= len(classificacao_nice_traducao):
                str_traducao = "NULL"
            else:
                str_traducao = f"'{classificacao_nice_traducao[i]}'"
            if i >= len(classificacao_nice_status):
                str_status = "NULL"
            else:
                str_status = f"'{classificacao_nice_status[i]}'"
            cmd = f"INSERT IGNORE INTO classes_nice (id, numero, codigo, especificacao, traducao, status, data) VALUES (NULL, '{numero}', {str_codigo}, {str_especificacao}, {str_traducao}, {str_status}, '{data}');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        classificacao_nice_codigo = []

    if classificacao_nacional_classe:
        for i in range(len(classificacao_nacional_classe)):
            if i >= len(classificacao_nacional_classe):
                str_classe = "NULL"
            else:
                str_classe = f"'{classificacao_nacional_classe[i]}'"
            if i >= len(classificacao_nacional_especificacao):
                str_especificacao = "NULL"
            else:
                classificacao_nacional_especificacao[i] = classificacao_nacional_especificacao[i].replace("'", " ").rstrip('\\')
                str_especificacao = f"'{classificacao_nacional_especificacao[i]}'"

            if i >= len(classificacao_nacional_subclasse):
                str_subclasse = "NULL"
            else:
                str_subclasse = f"'{classificacao_nacional_subclasse[i]}'"
            cmd = f"INSERT IGNORE INTO classes_nacional (id, numero, codigo_classe, especificacao, codigo_subclasse, data) VALUES (NULL, '{numero}', {str_classe}, {str_especificacao}, {str_subclasse}, '{data}');"
            print(cmd)
            arquivo.write(cmd + "\n")
            if gravar: 
                cursor.execute(cmd)
                conexao.commit()
        classificacao_nacional_classe = []

    if texto_prioridade:
        prioridades=', '.join(texto_prioridade)
        cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{prioridades}', 'prioridade');"
        print(cmd)
        texto_prioridade = ''
        arquivo.write(cmd + "\n")
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

    if apostila != '':
        apostila = apostila.replace("'", " ").rstrip('\\')
        cmd = f"INSERT IGNORE INTO revistas4ipas (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{apostila}', 'apostila');"
        print(cmd)
        arquivo.write(cmd + "\n")
        apostila = ''
        if gravar: 
            cursor.execute(cmd)
            conexao.commit()

arquivo.close()



IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [7]:
import xml.etree.ElementTree as ET

def list_unique_tags(xml_file):
    # Parse o arquivo XML
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Use um conjunto para armazenar as tags únicas
    unique_tags = set()
    
    # Função recursiva para percorrer as tags
    def traverse(element):
        unique_tags.add(element.tag)
        for child in element:
            traverse(child)
    
    # Percorre o elemento raiz
    traverse(root)
    
    # Retorna as tags únicas
    return unique_tags

# Substitua 'seu_arquivo.xml' pelo caminho do seu arquivo XML
tags = list_unique_tags(filename)

print("Tags únicas encontradas:")
for tag in sorted(tags):
    print(tag)

Tags únicas encontradas:
apostila
cedente
cedentes
cessionario
cessionarios
classe-nacional
classe-nice
classe-vienna
classes-vienna
dados-de-madri
despacho
despachos
especificacao
lista-classe-nice
marca
nome
prioridade
prioridade-unionista
processo
procurador
protocolo
requerente
revista
sobrestador
sobrestadores
status
sub-classe-nacional
sub-classes-nacional
texto-complementar
texto-sobrestamento
titular
titulares
traducao-especificacao


In [561]:
import xml.etree.ElementTree as ET

def list_unique_tags_and_attributes(xml_file):
    # Parse o arquivo XML
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Use conjuntos para armazenar tags e atributos únicos
    unique_tags = set()
    unique_attributes = set()
    
    # Função recursiva para percorrer as tags
    def traverse(element):
        # Adiciona a tag ao conjunto
        unique_tags.add(element.tag)
        # Adiciona os atributos ao conjunto
        unique_attributes.update(element.attrib.keys())
        # Percorre os filhos
        for child in element:
            traverse(child)
    
    # Percorre o elemento raiz
    traverse(root)
    
    # Retorna as tags e atributos únicos
    return unique_tags, unique_attributes

# Substitua 'seu_arquivo.xml' pelo caminho do seu arquivo XML
tags, attributes = list_unique_tags_and_attributes(filename)

print("Tags únicas encontradas:")
for tag in sorted(tags):
    print(tag)

print("\nAtributos únicos encontrados:")
for attribute in sorted(attributes):
    print(attribute)


Tags únicas encontradas:
apostila
cessionario
classe-nacional
classe-nice
classe-vienna
classes-vienna
despacho
despachos
especificacao
marca
nome
prioridade
prioridade-unionista
processo
procurador
protocolo
requerente
revista
sobrestador
sobrestadores
sub-classe-nacional
sub-classes-nacional
texto-complementar
titular
titulares

Atributos únicos encontrados:
apresentacao
codigo
codigoServico
data
data-concessao
data-deposito
data-vigencia
edicao
marca
natureza
nome-razao-social
numero
pais
processo
uf


In [7]:
# Iniciar carga da revista de Contratos

def LerFilename(revista):
    comando = f'select * from rpis_lidas where rpi="{revista}"'
    cursor.execute(comando)
    resultado = cursor.fetchall()
    print(len(resultado))
    for row in resultado:
        print(row[0])
        data_revista = row[1]
    df = pd.DataFrame(resultado)
    data = df[1].astype("string")
    str_data_revista = str(data_revista)
    d=datetime.datetime.strptime(str_data_revista, "%Y-%m-%d")
    #print(d.year)
    aux = str(d.strftime("%d%m%Y"))
    filename = f'revistas/Contratos_{revista}_{aux}.xml'
    #filename = f'revistas/Contratos_{revista}.xml'
    return filename, str_data_revista

revista = 2883
#filename, data_revista = LerFilename(revista)
aux = '07042026'
filename = f'revistas/Contratos_{revista}_{aux}.xml'
data_revista = '2026-04-07'
print(filename)
print(data_revista)

revistas/Contratos_2883_07042026.xml
2026-04-07


In [8]:
def ConverteNumero(x):
    s = str(x)
    s = s.replace("BR","")
    s = s.replace(" ","")
    pos = s.find('-')
    return s[0:pos]

In [9]:
def ConverteData(data):
    d=datetime.datetime.strptime(data, "%d/%m/%Y")
    return f'{d.year:04}-{d.month:02}-{d.day:02}'

In [10]:
def LerRPI(filename,data_revista):
    file = open(file=filename,mode='r',encoding="utf8")
    file_content_raw=file.read()
    text1=re.compile("<despacho>")
    file_content=text1.split(file_content_raw)
    file.close()
    texto=file_content[0]
    t = re.compile(r'\d{2}/../\d{4}')
    check = t.findall(texto)
    data = ''.join(check)
    data=ConverteData(data)
    if data_revista == data:
        print("Data da RPI "+data_revista+" confere com a data gravada dentro do XML "+data) 
    else:
        print("Data da RPI "+data_revista+" não confere com a data gravada dentro do XML "+data) 
    return file_content

file_content = LerRPI(filename,data_revista)
print("Total registros: "+str(len(file_content)-1))

Data da RPI 2026-04-07 confere com a data gravada dentro do XML 2026-04-07
Total registros: 55


In [11]:
codigo_tag = re.compile('<codigo>(.*?)</codigo>') # <codigo>130</codigo>
titulo_tag = re.compile('<titulo>(.*?)</titulo>') # <titulo>Processo indeferido</titulo>
data_protocolo_tag = re.compile("<dataProtocolo inid=\"De\">(.*?)</dataProtocolo>") # <dataProtocolo inid="De">12/11/2024</dataProtocolo>
nome_completo_ce_tag = re.compile("<nomeCompleto inid=\"Ce\">(.*?)</nome-completo>") 
nome_completo_cs_tag = re.compile("<nomeCompleto inid=\"Cs\">(.*?)</nome-completo>") 
nome_completo_rq_tag = re.compile("<nomeCompleto inid=\"Rq\">(.*?)</nome-completo>") 

numero_Np_tag = re.compile("<numero inid=\"Np\">(.*?)</numero>") # <numero inid="Np">BR 70 2015 000603-1</numero>
numero_Pt_tag = re.compile("<numero inid=\"Pt\">(.*?)</numero>") 
numero_Ca_tag = re.compile("<numero inid=\"Ca\">(.*?)</numero>") 

codigo_list, titulo_list, numero_peticao_list, requerente_list, data_protocolo_list, numero_list, cedente_nome_list, cessionaria_nome_list, pais_cedente_list, pais_cessionaria_list, cessionaria_setor_list, numero_ompi_list, data_ompi_list, classe_internacional_list, classe_nacional_list, sigla_prioridade_list, numero_prioridade_list, data_prioridade_list, prioridade_list, prioridade_interna_list, data_fase_nacional_list, titulo_pedido_list, inventor_nome_completo_list, divisao_pedido_list, data_divisao_pedido_list, data_rpi_list, data_concessao_list, numero_certificado_list, naturezaDocumento_list, textoObjeto_list, siglaCategoria_list, descricaoMoeda_list, valorContrato_list, prazoContrato_list, prazoVigenciaPI_list, observacao_list = ([] for i in range(36))

#arquivo = open("comandosct.sql", "w", encoding="utf-8")

for line in file_content:
    
    codigo=codigo_tag.findall(line)
    titulo=titulo_tag.findall(line)

            
    numero_certificado=""
    naturezaDocumento=""
    textoObjeto=""
    siglaCategoria=""
    descricaoMoeda=""
    valorContrato=""
    prazoContrato=""
    prazoVigenciaPI=""
    observacao=""
    
    data_deposito=None
    numero=None
    nome_completo=None
    titulo_pedido=""
    data_rpi=" "
    data_protocolo = ""
    numero_peticao = ""
    requerente = ""
    cedente_nome = ""
    cessionaria_nome = ""
    pais_cedente = ""
    pais_cessionaria = ""
    cessionaria_setor = ""
    
    processos=re.findall("<processo-contrato>[\s\S]*</processo-contrato>",line)
    for processo in processos:
        data_protocolo=data_protocolo_tag.findall(processo)
        data_protocolo=', '.join(data_protocolo) # converte lista em string
        numero=numero_Np_tag.findall(processo)
        numero = ', '.join(numero)

        #peticoes
        peticoes_lista=re.findall("<peticoes>[\s\S]*</peticoes>",processo)
        for peticao_lista in peticoes_lista:
            peticoes=re.findall("<peticao>[\s\S]*</peticao>",peticao_lista)
            for peticao in peticoes:
                numeros=re.findall("<numero inid=\"Pt\">[\s\S]*</numero>",peticao)
                for numero_peticao in numeros:
                    n=re.findall("<numero inid=\"Pt\">(.*?)</numero>",numero_peticao)
                    numero_peticao = '; '.join(n)

                requerentes=re.findall("<requerente>[\s\S]*</requerente>",peticao)
                for r in requerentes:
                    n=re.findall("<nomeCompleto inid=\"Rq\">(.*?)</nomeCompleto>",r)
                    requerente = '; '.join(n)

        #cedentes
        cedentes_lista=re.findall("<cedentes>[\s\S]*</cedentes>",processo)
        for cedente_lista in cedentes_lista:
            cedentes=re.findall("<cedente>[\s\S]*</cedente>",cedente_lista)
            for cedente in cedentes:
                #print(cedente)
                #break
                cedente_nome=re.findall("<nomeCompleto inid=\"Ce\">(.*?)</nomeCompleto>",cedente)
                cedente_nome = '; '.join(cedente_nome)
                enderecos=re.findall("<endereco>[\s\S]*</endereco>",cedente)
                for endereco in enderecos:
                    #print(endereco)
                    #break
                    paises=re.findall("<pais>[\s\S]*</pais>",endereco)
                    for pais in paises:
                        #print(pais)
                        #break
                        pais_cedente=re.findall("<nome inid=\"Pe\">(.*?)</nome>",pais)
                        pais_cedente = '; '.join(pais_cedente)

        #cessionarias
        cessionarias_lista=re.findall("<cessionarias>[\s\S]*</cessionarias>",processo)
        for cessionaria_lista in cessionarias_lista:
            cessionarias=re.findall("<cessionaria>[\s\S]*</cessionaria>",cessionaria_lista)
            for cessionaria in cessionarias:
                cessionaria_nome=re.findall("<nomeCompleto inid=\"Cs\">(.*?)</nomeCompleto>",cessionaria)
                cessionaria_nome = '; '.join(cessionaria_nome)
                cessionaria_setor=re.findall("<setor inid=\"Se\">(.*?)</setor>",cessionaria)
                cessionaria_setor = '; '.join(cessionaria_setor)
                enderecos=re.findall("<endereco>[\s\S]*</endereco>",cessionaria)
                for endereco in enderecos:
                    #print(endereco)
                    #break
                    paises=re.findall("<pais>[\s\S]*</pais>",endereco)
                    for pais in paises:
                        #print(pais)
                        #break
                        pais_cessionaria=re.findall("<nome inid=\"Ps\">(.*?)</nome>",pais)
                        pais_cessionaria = '; '.join(pais_cessionaria)

        #certificados
        certificados_lista=re.findall("<certificados>[\s\S]*</certificados>",processo)
        for certificado_lista in certificados_lista:
            certificados=re.findall("<certificado>[\s\S]*</certificado>",certificado_lista)
            for certificado in certificados:
                numero_certificado=re.findall("<numero inid=\"Ca\">(.*?)</numero>",certificado)
                numero_certificado = '; '.join(numero_certificado)
                naturezaDocumento=re.findall("<naturezaDocumento inid=\"Nd\">(.*?)</naturezaDocumento>",certificado)
                naturezaDocumento = '; '.join(naturezaDocumento)
                textoObjeto=re.findall("<textoObjeto inid=\"Ob\">(.*?)</textoObjeto>",certificado)
                textoObjeto = '; '.join(textoObjeto)
                siglaCategoria=re.findall("<siglaCategoria inid=\"Mc\">(.*?)</siglaCategoria>",certificado)
                siglaCategoria = '; '.join(siglaCategoria)
                descricaoMoeda=re.findall("<descricaoMoeda inid=\"Mo\">(.*?)</descricaoMoeda>",certificado)
                descricaoMoeda = '; '.join(descricaoMoeda)
                valorContrato=re.findall("<valorContrato inid=\"Va\">(.*?)</valorContrato>",certificado)
                valorContrato = '; '.join(valorContrato)
                prazoContrato=re.findall("<prazoContrato inid=\"Pz\">(.*?)</prazoContrato>",certificado)
                prazoContrato = '; '.join(prazoContrato)
                prazoVigenciaPI=re.findall("<prazoVigenciaPI inid=\"Pv\">(.*?)</prazoVigenciaPI>",certificado)
                prazoVigenciaPI = '; '.join(prazoVigenciaPI)
                observacao=re.findall("<observacao inid=\"Oc\">(.*?)</observacao>",certificado)
                observacao = '; '.join(observacao)
                        
    # checking length of gid is not equal to 0 then do append to all the lists
    if len(codigo)!=0:                             
        codigo_list.append(codigo[0])
        titulo_list.append(titulo[0])
        data_protocolo_list.append(data_protocolo)
        numero_list.append(numero)
        requerente_list.append(requerente)
        numero_peticao_list.append(numero_peticao)
        data_rpi_list.append(data_rpi[0])
        cedente_nome_list.append(cedente_nome)
        pais_cedente_list.append(pais_cedente)
        cessionaria_nome_list.append(cessionaria_nome)
        pais_cessionaria_list.append(pais_cessionaria)
        cessionaria_setor_list.append(cessionaria_setor)
        numero_certificado_list.append(numero_certificado)
        naturezaDocumento_list.append(naturezaDocumento)
        textoObjeto_list.append(textoObjeto)
        siglaCategoria_list.append(siglaCategoria)
        descricaoMoeda_list.append(descricaoMoeda)
        valorContrato_list.append(valorContrato)
        prazoContrato_list.append(prazoContrato)
        prazoVigenciaPI_list.append(prazoVigenciaPI)
        observacao_list.append(observacao)
        


In [12]:
data_frame = pd.DataFrame(
    {'despacho': codigo_list,
     'numero': numero_list,
     'data_protocolo': data_protocolo_list,
     'requerente': requerente_list,
     'numero_peticao': numero_peticao_list,
     'texto_despacho': titulo_list,
     'cedente': cedente_nome_list,
     'pais_cedente':pais_cedente_list,
     'cessionaria': cessionaria_nome_list,
     'pais_cessionaria': pais_cessionaria_list,
     'setor': cessionaria_setor_list,
     'certificado': numero_certificado_list,
     'naturezaDocumento': naturezaDocumento_list,
     'textoObjeto': textoObjeto_list,
     'siglaCategoria': siglaCategoria_list,
     'descricaoMoeda': descricaoMoeda_list,
     'valorContrato': valorContrato_list,
     'prazoContrato': prazoContrato_list,
     'prazoVigenciaPI': prazoVigenciaPI_list,
     'observacao': observacao_list
    })

In [13]:
data_frame["numero_new"] = data_frame["numero"].apply(ConverteNumero)
# https://pt.stackoverflow.com/questions/564421/aplicar-fun%c3%a7%c3%a3o-lamba-para-uma-subtra%c3%a7%c3%a3o-e-soma-python/564425#564425

In [14]:
print(len(data_frame.index))
data_frame[["despacho","numero_new","data_protocolo","requerente","numero_peticao","texto_despacho","cedente","pais_cedente","cessionaria","pais_cessionaria","setor","certificado","naturezaDocumento","textoObjeto","siglaCategoria","descricaoMoeda","valorContrato","prazoContrato","prazoVigenciaPI","observacao"]]
#df1 = data_frame.loc[data_frame['numero_new']=='702014000030']
#df1.head(1)
#data_frame.loc[89]
data_frame.head(1)

55


,despacho,numero,data_protocolo,requerente,numero_peticao,texto_despacho,cedente,pais_cedente,cessionaria,pais_cessionaria,...,certificado,naturezaDocumento,textoObjeto,siglaCategoria,descricaoMoeda,valorContrato,prazoContrato,prazoVigenciaPI,observacao,numero_new
0,145,BR 70 2022 000081-9,,,,Exigência,INTERNATIONAL CONTAINER TERMINAL SERVICES INC.,,TECON SUAPE S.A.,,...,,,,,,,,,,702022000081


In [15]:
for i, infos in data_frame.iterrows():
    despacho = infos.despacho
    numero = infos.numero_new
    data_protocolo = infos.data_protocolo
    if data_protocolo is not None and data_protocolo!='':
        data_protocolo = ConverteData(infos.data_protocolo)
    else:
        data_protocolo = None
    requerente = infos.requerente
    requerente = requerente.replace("'", "\\'")
    numero_peticao = infos.numero_peticao
    texto_despacho = infos.texto_despacho
    cedente = infos.cedente
    cedente = cedente.replace("'", "\\'")
    pais_cedente = infos.pais_cedente
    cessionaria = infos.cessionaria
    cessionaria = cessionaria.replace("'", "\\'")
    pais_cessionaria = infos.pais_cessionaria
    setor = infos.setor
    setor = setor.replace("'", "\\'")
    certificado = infos.certificado
    naturezaDocumento = infos.naturezaDocumento
    textoObjeto = infos.textoObjeto
    textoObjeto = textoObjeto.replace("'", "\\'")
    siglaCategoria = infos.siglaCategoria
    descricaoMoeda = infos.descricaoMoeda
    valorContrato = infos.valorContrato
    prazoContrato = infos.prazoContrato
    prazoVigenciaPI = infos.prazoVigenciaPI
    observacao = infos.observacao
    observacao = observacao.replace("'", "\\'")
   
    comando = f"INSERT IGNORE INTO arquivadosct (id, data, despacho, numero, data_protocolo, requerente, numero_peticao, texto_despacho, cedente, pais_cedente, cessionaria, pais_cessionaria, setor, certificado, naturezaDocumento, textoObjeto, siglaCategoria, descricaoMoeda, valorContrato, prazoContrato, prazoVigenciaPI, observacao) VALUES (NULL, '{data_revista}', '{despacho}', '{numero}', '{data_protocolo}', '{requerente}', '{numero_peticao}','{texto_despacho}', '{cedente}', '{pais_cedente}', '{cessionaria}', '{pais_cessionaria}', '{setor}', '{certificado}', '{naturezaDocumento}', '{textoObjeto}', '{siglaCategoria}','{descricaoMoeda}', '{valorContrato}', '{prazoContrato}', '{prazoVigenciaPI}', '{observacao}');"
    if data_protocolo is None:
        comando = f"INSERT IGNORE INTO arquivadosct (id, data, despacho, numero, data_protocolo, requerente, numero_peticao, texto_despacho, cedente, pais_cedente, cessionaria, pais_cessionaria, setor, certificado, naturezaDocumento, textoObjeto, siglaCategoria, descricaoMoeda, valorContrato, prazoContrato, prazoVigenciaPI, observacao) VALUES (NULL, '{data_revista}', '{despacho}', '{numero}', NULL, '{requerente}', '{numero_peticao}','{texto_despacho}', '{cedente}', '{pais_cedente}', '{cessionaria}', '{pais_cessionaria}', '{setor}', '{certificado}', '{naturezaDocumento}', '{textoObjeto}', '{siglaCategoria}','{descricaoMoeda}', '{valorContrato}', '{prazoContrato}', '{prazoVigenciaPI}', '{observacao}');"

    print(comando)
    #break
    #try:
        #cursor.execute(comando)
        #conexao.commit()
        #arquivo.write(comando + "\n")
    #except (mysql.connector.Error, mysql.connector.Warning) as e:
    #    print(e)
    #    print(str(numero))

print("Comando encerrado!")
#arquivado.close()

INSERT IGNORE INTO arquivadosct (id, data, despacho, numero, data_protocolo, requerente, numero_peticao, texto_despacho, cedente, pais_cedente, cessionaria, pais_cessionaria, setor, certificado, naturezaDocumento, textoObjeto, siglaCategoria, descricaoMoeda, valorContrato, prazoContrato, prazoVigenciaPI, observacao) VALUES (NULL, '2026-04-07', '145', '702022000081', NULL, '', '','Exigência', 'INTERNATIONAL CONTAINER TERMINAL SERVICES INC.', '', 'TECON SUAPE S.A.', '', '', '', '', '', '','', '', '', '', '');
INSERT IGNORE INTO arquivadosct (id, data, despacho, numero, data_protocolo, requerente, numero_peticao, texto_despacho, cedente, pais_cedente, cessionaria, pais_cessionaria, setor, certificado, naturezaDocumento, textoObjeto, siglaCategoria, descricaoMoeda, valorContrato, prazoContrato, prazoVigenciaPI, observacao) VALUES (NULL, '2026-04-07', '145', '702025000245', NULL, '', '','Exigência', 'SHENYANG ALUMINUM AND MAGNESIUM ENGINEERING AND RESEARCH INSTITUTE COMPANY LIMITED', '', 'C

In [17]:
#confere total de registros gravados
comando = f'select count(*) as total from arquivadosct where data="{data_revista}"'
cursor.execute(comando)
resultado = cursor.fetchall()
for row in resultado:
    if len(data_frame.index) == row[0]:
        print("Sucesso na gravação do total de "+str(row[0])+" registros !")
    else:
        print(len(data_frame.index))



Sucesso na gravação do total de 86 registros !


In [41]:
def numOfDays(date1, date2):
    return (date2-date1).days

#confere média de tempo de registro
soma = 0
total = 0
comando = f'select * from arquivadosct where despacho="350" and year(data)=2022'
cursor.execute(comando)
resultado = cursor.fetchall()
for row in resultado:
    print(row[3])
    print("data concessão= "+str(row[1]))
    d=datetime.datetime.strptime(str(row[1]), "%Y-%m-%d")
    date1 = datetime.date(d.year, d.month, d.day)
    print("data protocolo= "+str(row[4]))
    d=datetime.datetime.strptime(str(row[4]), "%Y-%m-%d")
    date2 = datetime.date(d.year, d.month, d.day)
    print(numOfDays(date2, date1), "dias")
    soma = soma + numOfDays(date2, date1)
    total = total + 1
    print("\n")
    
print("Fim de processamento")
media = soma/total
print(f'Tempo médio de dias para concessão: {media:.2f}')


07026
data concessão= 2022-01-04
data protocolo= 2021-12-10
25 dias


11067
data concessão= 2022-01-04
data protocolo= 2021-02-12
326 dias


11119
data concessão= 2022-01-04
data protocolo= 2021-05-14
235 dias


12099
data concessão= 2022-01-04
data protocolo= 2021-12-09
26 dias


702015000285
data concessão= 2022-01-04
data protocolo= 2021-11-04
61 dias


702017000024
data concessão= 2022-01-04
data protocolo= 2021-12-21
14 dias


702017000221
data concessão= 2022-01-04
data protocolo= 2021-12-03
32 dias


702018000057
data concessão= 2022-01-04
data protocolo= 2021-12-03
32 dias


702018000058
data concessão= 2022-01-04
data protocolo= 2021-12-03
32 dias


702019000026
data concessão= 2022-01-04
data protocolo= 2021-12-03
32 dias


702019000654
data concessão= 2022-01-04
data protocolo= 2021-12-10
25 dias


702020000491
data concessão= 2022-01-04
data protocolo= 2021-12-20
15 dias


702021000228
data concessão= 2022-01-04
data protocolo= 2021-05-17
232 dias


702021000387
data conces

In [24]:
def numOfDays(date1, date2):
    return (date2-date1).days

#confere média de tempo de registro
soma = 0
total = 0
for ano in range(1998,2025):
    soma = 0
    total = 0
    comando = f"select * from arquivados where (numero like 'PI%' or numero like '1%') and despacho='16.1' and year(data)={ano}"
    cursor.execute(comando)
    resultado = cursor.fetchall()
    for row in resultado:
        # print(row[2]) # 
        numero = str(row[2])
        # print("data concessão= "+str(row[3]))
        d=datetime.datetime.strptime(str(row[3]), "%Y-%m-%d")
        date1 = datetime.date(d.year, d.month, d.day)

        comando2 = f"select * from publicados where numero='{numero}'"
        cursor.execute(comando2)
        resultado2 = cursor.fetchall()
        data_nacional = None
        for row2 in resultado2:
            data_nacional = str(row2[9])

        if data_nacional is not None and data_nacional.lower() != 'none':
            # print("data nacional= "+data_nacional)
            d=datetime.datetime.strptime(data_nacional, "%Y-%m-%d")
            date2 = datetime.date(d.year, d.month, d.day)
            # print(numOfDays(date2, date1), "dias")
            soma = soma + numOfDays(date2, date1)
            total = total + 1
            #print("\n")
            #break;
    
    media = round((soma/total)/365,2)
    print(f'Tempo médio de anos para concessão {ano}: {media:.2f}')

print("Fim de processamento")


Tempo médio de anos para concessão 1998: 6.92
Tempo médio de anos para concessão 1999: 6.97
Tempo médio de anos para concessão 2000: 5.60
Tempo médio de anos para concessão 2001: 5.40
Tempo médio de anos para concessão 2002: 6.30
Tempo médio de anos para concessão 2003: 6.22
Tempo médio de anos para concessão 2004: 6.25
Tempo médio de anos para concessão 2005: 7.47
Tempo médio de anos para concessão 2006: 7.40
Tempo médio de anos para concessão 2007: 7.46
Tempo médio de anos para concessão 2008: 8.13
Tempo médio de anos para concessão 2009: 8.63
Tempo médio de anos para concessão 2010: 9.01
Tempo médio de anos para concessão 2011: 8.96
Tempo médio de anos para concessão 2012: 9.10
Tempo médio de anos para concessão 2013: 9.71
Tempo médio de anos para concessão 2014: 9.64
Tempo médio de anos para concessão 2015: 9.97
Tempo médio de anos para concessão 2016: 9.95
Tempo médio de anos para concessão 2017: 9.66
Tempo médio de anos para concessão 2018: 9.25
Tempo médio de anos para concessão

In [ ]:
CREATE TABLE `arquivadosct` ( `id` int(11) NULL AUTO_INCREMENT, `despacho` char(30) NULL, `numero` char(15) NULL, `data_protocolo` date NULL, `requerente` TEXT NULL, `numero_peticao` char(15) NULL, `texto_despacho` TEXT NULL, `cedente` TEXT NULL, `pais_cedente` char(30) NULL, `cessionaria` TEXT NULL, `pais_cessionaria` char(30) NULL, `setor` TEXT NULL, `certificado` char(30) NULL, `naturezaDocumento` char(30) NULL, `textoObjeto` TEXT NULL, `siglaCategoria` char(30) NULL, `descricaoMoeda` char(15) NULL, `valorContrato` char(255) NULL, `prazoContrato` char(255) NULL, `prazoVigenciaPI` TEXT NULL, `observacao` TEXT NULL ) ENGINE=InnoDB DEFAULT CHARSET=latin1 COMMENT='todos os despachos publicados nas RPIs'; 
ALTER TABLE `arquivadosct` ADD PRIMARY KEY (`id`); ALTER TABLE `arquivadosxml` MODIFY `id` int(11) NOT NULL AUTO_INCREMENT; 

In [21]:
# Rotina para testar carga de revistas antigas
# verifica tabelas arquivados_ipas_old
# publicados_ipas_old
# revistas4ipas_old
# tem que rodar as primeiras linhas deste programa sinergias3marcas.ipynb para ler as funções e a base de dados localhost do xampp
# Etapa 1: gere comandos.sql rodando este programa
# Etapa 2: importe comandos.sql no hostgator para atualizar arquivados_ipas_old
# Etapa 3: rode https://cientistaspatentes.com.br/central/control.php?action=180
# Etapa 4: copie os comandos e rode na aba SQL do hostgator para atualizar arquivados_ipas com os despachos faltantes

revista =  2771                   # 2837 já com algoritmo novo até 2244 ou seja 593 revistas ! feito 2837-2837
filename, data_revista = LerFilename(revista)
#filename = 'revistas/teste.xml'
print(filename)
print(data_revista)
print("File: " + filename)
file_content = LerRPI(filename,data_revista)
print(" Total registros: " + str(len(file_content)))
print(data_revista)


1
2771
revistas/RM2771.xml
2024-02-15
File: revistas/RM2771.xml
15/02/2024
Número de registros de processos: 17486
Número de processos encontrados: 17486
Data da RPI 2024-02-15 confere com a data gravada dentro do XML 2024-02-15
 Total registros: 17486
2024-02-15


In [22]:

procurador_tag = re.compile('<procurador>(.*?)</procurador>', re.DOTALL) # <procurador>Bibbiana Bertolaccini Vasconcelos</procurador>
nome_tag = re.compile('<nome>(.*?)</nome>', re.DOTALL) 
especificacao_tag = re.compile('<especificacao>(.*?)</especificacao>', re.DOTALL) 
traducao_especificacao_tag = re.compile('<traducao-especificacao>(.*?)</traducao-especificacao>', re.DOTALL) 
status_tag = re.compile('<status>(.*?)</status>', re.DOTALL)
texto_complementar_tag = re.compile('<texto-complementar>(.*?)</texto-complementar>', re.DOTALL)
texto_sobrestamento_tag = re.compile('<texto-sobrestamento>(.*?)</texto-sobrestamento>', re.DOTALL)
apostila_tag = re.compile('<apostila>(.*?)</apostila>', re.DOTALL)
cedentes_tag = re.compile('<cedentes>(.*?)</cedentes>', re.DOTALL) 
cessionarios_tag = re.compile('<cessionarios>(.*?)</cessionarios>', re.DOTALL) 

nome_razao_social_pattern = re.compile(r'nome-razao-social="([^"]+)"', re.DOTALL)
pais_pattern = re.compile(r'pais="([^"]+)"', re.DOTALL)
uf_pattern = re.compile(r'uf="([^"]+)"', re.DOTALL)
codigo_pattern = re.compile(r'codigo="IPAS(\d+)"', re.DOTALL)
nome_pattern = re.compile(r'nome="([^"]+)"', re.DOTALL)
apresentacao_pattern = re.compile(r'apresentacao="([^"]+)"', re.DOTALL)
natureza_pattern = re.compile(r'natureza="([^"]+)"', re.DOTALL)
codigo_vienna_pattern = re.compile(r'codigo="([^"]+)"', re.DOTALL)
edicao_vienna_pattern = re.compile(r'edicao="(\d+)"', re.DOTALL)
codigo_nice_pattern = re.compile(r'codigo="(\d+)"', re.DOTALL)
numero_pattern = re.compile(r'processo numero="(\d+)"', re.DOTALL)
data_deposito_pattern = re.compile(r'data-deposito="([^"]+)"', re.DOTALL)
data_prioridade_pattern = re.compile(r'data="([^"]+)"', re.DOTALL)
numero_prioridade_pattern = re.compile(r'numero="([^"]+)"', re.DOTALL)
pais_prioridade_pattern = re.compile(r'pais="([^"]+)"', re.DOTALL)
protocolo_numero_pattern = re.compile(r'numero="([^"]+)"', re.DOTALL)
protocolo_data_pattern = re.compile(r'data="([^"]+)"', re.DOTALL)
protocolo_codigoServico_pattern = re.compile(r'codigoServico="([^"]+)"', re.DOTALL)
protocolo_nome_razao_social_pattern = re.compile(r'nome-razao-social="([^"]+)"', re.DOTALL)
protocolo_pais_pattern = re.compile(r'pais="([^"]+)"', re.DOTALL)
protocolo_uf_pattern = re.compile(r'uf="([^"]+)"', re.DOTALL)
data_concessao_pattern = re.compile(r'data-concessao="([^"]+)"', re.DOTALL)
data_vigencia_pattern = re.compile(r'data-vigencia="([^"]+)"', re.DOTALL)
codigo_classe_nacional_pattern = re.compile(r'<classe-nacional codigo="(\d+)"', re.DOTALL)
codigo_subclasse_nacional_pattern = re.compile(r'<sub-classe-nacional codigo="(\d+)"', re.DOTALL)
numero_madri_pattern = re.compile(r'numero-inscricao-internacional="([^"]+)"', re.DOTALL)
data_recebimento_inpi_pattern = re.compile(r'data-recebimento-inpi="([^"]+)"', re.DOTALL)
sobrestadores_processo_pattern = re.compile(r'processo="([^"]+)"', re.DOTALL)
sobrestadores_marca_pattern = re.compile(r'marca="([^"]+)"', re.DOTALL)

arquivo = open("comandos.sql", "w", encoding="utf-8")
cmd = 'TRUNCATE TABLE arquivados_ipas_old;'
print(cmd)
arquivo.write(cmd + "\n")
cmd = 'TRUNCATE TABLE publicados_ipas_old;'
print(cmd)
arquivo.write(cmd + "\n")
cmd = 'TRUNCATE TABLE revistas4ipas_old;'
print(cmd)
arquivo.write(cmd + "\n")

for line in file_content:
    
    numero = ''
    data_deposito = None
    data_concessao = None
    data_vigencia = None
    numero_internacional = ''
    data_recebimento = None
    codigo_ipas = ''
    nome_ipas = ''
    texto_complementar = []
    texto_sobrestamento = []
    texto_protocolo = []
    protocolo_numero = []
    protocolo_data = []
    protocolo_codigo_servico = []
    protocolo_procurador = []
    protocolo_nome_razao_social = []
    protocolo_pais = []
    protocolo_uf = []
    texto_cessionario = [] 
    texto_cedente = []
    cedente_nome = []
    cedente_pais = []
    cedente_uf = []
    cessionario_nome = []
    texto_titulares = []
    texto_sobrestador = []
    marca_natureza = ''
    marca_apresentacao = ''
    marca_nome = ''
    classificacao_vienna_codigo = []
    classificacao_vienna_edicao = []
    classificacao_nice_codigo = []
    classificacao_nice_especificacao = []
    classificacao_nice_traducao = []
    classificacao_nice_status = []
    classificacao_nacional_classe = []
    classificacao_nacional_especificacao = []
    classificacao_nacional_subclasse = []
    texto_prioridade = []
    
    print('\n')
    numero_matches = numero_pattern.findall(line)
    for numero in numero_matches:
        print(f"Número: {numero}")
    data_deposito_matches = data_deposito_pattern.findall(line)
    for data_deposito in data_deposito_matches:
        print(f"Data depósito: {ConverteData(data_deposito)}")
        data_deposito = ConverteData(data_deposito)
    data_concessao_matches = data_concessao_pattern.findall(line)
    for data_concessao in data_concessao_matches:
        print(f"Data concessão: {ConverteData(data_concessao)}")
        data_concessao = ConverteData(data_concessao)
    data_vigencia_matches = data_vigencia_pattern.findall(line)
    for data_vigencia in data_vigencia_matches:
        print(f"Data vigência: {ConverteData(data_vigencia)}")
        data_vigencia = ConverteData(data_vigencia)
    
    dados_de_madris=re.findall("<dados-de-madri [\s\S]*/>",line)
    for dados_de_madri in dados_de_madris:
        numero_matches = numero_madri_pattern.findall(dados_de_madri)
        data_recebimento_inpi_matches = data_recebimento_inpi_pattern.findall(dados_de_madri)
        i = 0
        for match in numero_matches:
            print(f"Número Inscrição Internacional: {match}")
            numero_internacional = match
            if i < len(data_recebimento_inpi_matches):
                print(f"Data recebimento INPI: {data_recebimento_inpi_matches[i]}")
                data_recebimento = ConverteData(data_recebimento_inpi_matches[i])
            i = i + 1

            
    nome_razao_social_pattern = re.compile(r'nome-razao-social="([^"]+)"', re.DOTALL)
    titulares=re.findall("<titulares>[\s\S]*</titulares>",line)
    for titular in titulares:
        nome_razao_social_matches = nome_razao_social_pattern.findall(titular)
        pais_matches = pais_pattern.findall(titular)
        uf_matches = uf_pattern.findall(titular)
        i = 0
        for match in nome_razao_social_matches:
            print(f"Titular Nome/Razão Social: {match}")
            ref = match
            if i < len(pais_matches):
                print(f"País: {pais_matches[i]}")
                ref = f"{match} [{pais_matches[i]}]"
            if i < len(uf_matches):
                print(f"UF: {uf_matches[i]}")
                ref = f"{match} [{pais_matches[i]}/{uf_matches[i]}]"
            texto_titulares.append(ref)
            i = i + 1

    sobrestadores=re.findall("<sobrestadores>[\s\S]*</sobrestadores>",line)
    for sobrestador in sobrestadores:
        processo_matches = sobrestadores_processo_pattern.findall(sobrestador)
        marca_matches = sobrestadores_marca_pattern.findall(sobrestador)
        i = 0
        for match in processo_matches:
            print(f"Sobrestador processo: {match}")
            ref = match
            if i < len(marca_matches):
                print(f"Marca: {marca_matches[i]}")
                ref = f"{match}, marca: {marca_matches[i]}"
            texto_sobrestador.append(ref)
            i = i + 1

    # <marca apresentacao="Figurativa" natureza="Produtos e/ou Serviço"/>
    marcas_autocontidos = re.findall(r'<marca apresentacao="([^"]+)" natureza="([^"]+)"\s*/>', line)
    for marca_apresentacao, marca_natureza in marcas_autocontidos:
        print(f"Marca apresentacao: {marca_apresentacao}")
        print(f"Marca apresentação: {marca_natureza}")

    marcas=re.findall("<marca [\s\S]*</marca>",line)
    for marca in marcas:
        apresentacao_matches = apresentacao_pattern.findall(marca)
        natureza_matches = natureza_pattern.findall(marca)
        i = 0
        for match in apresentacao_matches:
            print(f"apresentacao: {match}")
            marca_apresentacao = match
            if i < len(natureza_matches):
                print(f"natureza: {natureza_matches[i]}")
                marca_natureza = natureza_matches[i]
            i = i + 1
        nomes=nome_tag.findall(marca)
        for nome in nomes:
            print(f"nome: {nome}")
            marca_nome = nome

    classes_vienna=re.findall("<classes-vienna>[\s\S]*</classes-vienna>",line)
    for classe_vienna in classes_vienna:
        classes_vienna2=re.findall("<classe-vienna [\s\S]*/>",classe_vienna)
        for classe_vienna1 in classes_vienna2:
            codigo_vienna_matches = codigo_vienna_pattern.findall(classe_vienna1)
            edicao_vienna_matches = edicao_vienna_pattern.findall(classe_vienna1)
            i = 0
            for match in codigo_vienna_matches:
                print(f"Codigo: {match}")
                classificacao_vienna_codigo.append(match)
                if i < len(edicao_vienna_matches):
                    print(f"edicao: {edicao_vienna_matches[i]}")
                    classificacao_vienna_edicao.append(edicao_vienna_matches[i])
                i = i + 1

    listas_classe_nice=re.findall("<lista-classe-nice>[\s\S]*</lista-classe-nice>",line)
    for lista_classe_nice in listas_classe_nice:
        classes_nice=re.findall("<classe-nice [\s\S]*</classe-nice>",lista_classe_nice)
        for classe_nice in classes_nice:
            codigo_nice_matches = codigo_nice_pattern.findall(classe_nice)
            especificacao_matches=especificacao_tag.findall(classe_nice)
            traducao_especificacao_matches=traducao_especificacao_tag.findall(classe_nice)
            status_matches=status_tag.findall(classe_nice)
            i = 0
            for match in codigo_nice_matches:
                print(f"Codigo Nice: {match}")
                classificacao_nice_codigo.append(match)
                if i < len(especificacao_matches):
                    print(f"especificação: {especificacao_matches[i]}")
                    especificacao_matches[i] = especificacao_matches[i].replace("'", "''")
                    classificacao_nice_especificacao.append(especificacao_matches[i])
                if i < len(traducao_especificacao_matches):
                    print(f"tradução especificação: {traducao_especificacao_matches[i]}")
                    traducao_especificacao_matches[i] = traducao_especificacao_matches[i].replace("'", "''")
                    classificacao_nice_traducao.append(traducao_especificacao_matches[i])
                if i < len(status_matches):
                    print(f"status: {status_matches[i]}")
                    classificacao_nice_status.append(status_matches[i])
                i = i + 1

    classe_nacionals=re.findall("<classe-nacional[\s\S]*?</classe-nacional>",line)
    i = 0
    for classe_nacional in classe_nacionals:
        #print(f"teste {classe_nacional}")
        codigo_classe_nacional_matches = codigo_classe_nacional_pattern.findall(classe_nacional)
        for match in codigo_classe_nacional_matches:
            print(f"Codigo Classe nacional: {match}")
            classificacao_nacional_classe.append(match)
        especificacao_matches = especificacao_tag.findall(classe_nacional)
        for match in especificacao_matches:
            print(f"especificacao: {match}")
            match = match.replace("'", "''")
            classificacao_nacional_especificacao.append(match)
        subclasses_nacional=re.findall("<sub-classes-nacional>[\s\S]*</sub-classes-nacional>",classe_nacional)
        for subclasse_nacional in subclasses_nacional:
            codigo_matches = codigo_subclasse_nacional_pattern.findall(subclasse_nacional)
            classificacao_nacional_subclasse.append(', '.join(codigo_matches))
            for match in codigo_matches:
                print(f"Codigo subclasse {i}: {match}")
        i = i + 1
                
    prioridades_unionista=re.findall("<prioridade-unionista>[\s\S]*?</prioridade-unionista>",line)
    for prioridade_unionista in prioridades_unionista:
        data_prioridade_matches = data_prioridade_pattern.findall(prioridade_unionista)
        numero_prioridade_matches = numero_prioridade_pattern.findall(prioridade_unionista)
        pais_prioridade_matches = pais_prioridade_pattern.findall(prioridade_unionista)
        i = 0
        for match in data_prioridade_matches:
            print(f"Data prioridade: {match}")
            data_prioridade = match
            numero_prioridade = ''
            pais_prioridade = ''
            if i < len(numero_prioridade_matches):
                print(f"Número prioridade: {numero_prioridade_matches[i]}")
                numero_prioridade = numero_prioridade_matches[i]
            if i < len(pais_prioridade_matches):
                print(f"País prioridade: {pais_prioridade_matches[i]}")
                pais_prioridade = pais_prioridade_matches[i]
            texto_prioridade.append(f"[{pais_prioridade}] {numero_prioridade} de {data_prioridade}")
            i = i + 1

    apostila = ''
    apostilas=apostila_tag.findall(line)
    for apostila in apostilas:
        print(f"apostila: {apostila}")

    procurador = ''
    procuradores=procurador_tag.findall(line)
    if procuradores:  # Verifica se a lista não está vazia
        for i, procurador in enumerate(procuradores):
            if i == len(procuradores) - 1:  # Verifica se é o último elemento
                print(f"Procurador: {procurador}")
        procurador = ', '.join(procuradores)
    else:
        print("A lista de procuradores está vazia.")
        
       
    despachos_com_conteudo = re.findall(r'<despacho codigo="IPAS(\d+)" nome="([^"]+)">([\s\S]*?)</despacho>', line)
    for codigo_ipas, nome_ipas, despacho in despachos_com_conteudo:
        texto_complementar = []
        texto_sobrestamento = []
        texto_protocolo = []
        protocolo_numero = []
        protocolo_data = []
        protocolo_codigo_servico = []
        protocolo_procurador = []
        protocolo_nome_razao_social = []
        protocolo_pais = []
        protocolo_uf = []
        texto_protocolo = []
        cedente_nome = []
        cedente_pais = []
        cedente_uf = []
        texto_cedente = []
        cessionario_nome = []
        texto_cessionario = []

        print(f"Despacho Código: IPAS{codigo_ipas}")
        print(f"Nome: {nome_ipas}")
        data = data_revista
        cmd = f"INSERT INTO arquivados_ipas_old (id, codigo_ipas, numero, data, anulado, prmexame) VALUES (NULL, '{codigo_ipas}', '{numero}', '{data}', 0, 0);"
        print(cmd)
        arquivo.write(cmd + "\n")
        
        texto_complementar_matches = texto_complementar_tag.findall(despacho)
        for match in texto_complementar_matches:
            print(f"texto complementar: {match}")
            #texto_complementar = match
            texto_complementar.append(match)
        texto_sobrestamento_matches = texto_sobrestamento_tag.findall(despacho)
        for match in texto_sobrestamento_matches:
            print(f"texto sobrestamento: {match}")
            #texto_sobrestamento = match
            texto_sobrestamento.append(match)
        protocolos=re.findall("<protocolo [\s\S]*</protocolo>",despacho)
        for protocolo in protocolos:
            protocolo_numero_matches = protocolo_numero_pattern.findall(protocolo)
            protocolo_data_matches = protocolo_data_pattern.findall(protocolo)
            protocolo_codigoServico_matches = protocolo_codigoServico_pattern.findall(protocolo)
            i = 0
            for match in protocolo_numero_matches:
                print(f"Protocolo numero: {match}")
                protocolo_numero.append(match)
                if i < len(protocolo_data_matches):
                    print(f"data: {protocolo_data_matches[i]}")
                    protocolo_data.append(protocolo_data_matches[i])
                if i < len(protocolo_codigoServico_matches):
                    print(f"Código Serviço: {protocolo_codigoServico_matches[i]}")
                    protocolo_codigo_servico.append(protocolo_codigoServico_matches[i])
                i = i + 1
            procuradores=procurador_tag.findall(protocolo)
            i = 0
            for procurador in procuradores:
                print(f"procurador: {procurador}")
                protocolo_procurador.append(procurador)
            protocolo_nome_razao_social_matches=protocolo_nome_razao_social_pattern.findall(protocolo)
            protocolo_pais_matches=protocolo_pais_pattern.findall(protocolo)
            protocolo_uf_matches=protocolo_uf_pattern.findall(protocolo)
            i = 0
            for nome_razao_social in protocolo_nome_razao_social_matches:
                print(f"Nome razão social: {nome_razao_social}")
                protocolo_nome_razao_social.append(nome_razao_social)
                if i < len(protocolo_pais_matches):
                    print(f"País: {protocolo_pais_matches[i]}")
                    protocolo_pais.append(protocolo_pais_matches[i])
                if i < len(protocolo_uf_matches):
                    print(f"UF: {protocolo_uf_matches[i]}")
                    protocolo_uf.append(protocolo_uf_matches[i])
                i = i + 1
            i = 0
            for elemento in protocolo_numero:
                if i < len(protocolo_data):
                    str_data = protocolo_data[i]
                else:
                    str_data = ''
                if i < len(protocolo_codigo_servico):
                    str_codigo_servico = protocolo_codigo_servico[i]
                else:
                    str_codigo_servico = ''
                if i < len(protocolo_procurador):
                    str_procurador = protocolo_procurador[i]
                else:
                    str_procurador = ''
                if i < len(protocolo_nome_razao_social):
                    str_nome_razao_social = protocolo_nome_razao_social[i]
                else:
                    str_nome_razao_social = ''
                if i < len(protocolo_pais):
                    str_pais = protocolo_pais[i]
                else:
                    str_pais = ''
                if i < len(protocolo_uf):
                    str_uf = protocolo_uf[i]
                else:
                    str_uf = ''
                texto_protocolo.append(f"protocolo {elemento} de {str_data} e " \
                                  f"código de serviço {str_codigo_servico}, procurador: " \
                                  f"{str_procurador}, requerente: {str_nome_razao_social}" \
                                  f"[{str_pais}/{str_uf}]")
                i = i + 1
            cedentes=cedentes_tag.findall(protocolo)
            for cedente in cedentes:
                nome_razao_social_matches=protocolo_nome_razao_social_pattern.findall(cedente)
                pais_matches=protocolo_pais_pattern.findall(cedente)
                uf_matches=protocolo_uf_pattern.findall(cedente)
                i = 0
                for nome_razao_social in nome_razao_social_matches:
                    print(f"Cedente Nome razão social: {nome_razao_social}")
                    cedente_nome.append(nome_razao_social)
                    texto_nome = nome_razao_social
                    texto_pais = ''
                    texto_uf = ''
                    if i < len(pais_matches):
                        print(f"País: {pais_matches[i]}")
                        cedente_pais.append(pais_matches[i])
                        texto_pais = pais_matches[i]
                    if i < len(uf_matches):
                        print(f"UF: {uf_matches[i]}")
                        cedente_uf.append(uf_matches[i])
                        texto_uf = uf_matches[i]
                    texto_cedente.append(f"Cedente: {texto_nome} [{texto_pais}/{texto_uf}]")
                    i = i + 1
            cessionarios=cessionarios_tag.findall(protocolo)
            for cessionario in cessionarios:
                nome_razao_social_matches=protocolo_nome_razao_social_pattern.findall(cessionario)
                i = 0
                for nome_razao_social in nome_razao_social_matches:
                    print(f"Cessionário Nome razão social: {nome_razao_social}")
                    cessionario_nome.append(nome_razao_social)
                    texto_cessionario.append(f"Cessionário: {cessionario_nome[i]}")
                    i = i + 1

        if numero_internacional != '':
            cmd = f"INSERT IGNORE INTO madri_old (numero, numero_internacional, data, data_recebimento) VALUES ('{numero}', '{numero_internacional}', '{data}', '{data_recebimento}');"
            print(cmd)
            numero_internacional = ''
            # arquivo.write(cmd + "\n")
        if nome_ipas != '':
            cmd = f"INSERT IGNORE INTO ipas_old (codigo_ipas, nome_ipas) VALUES ('{codigo_ipas}', '{nome_ipas}');"
            print(cmd)
            nome_ipas = ''
            #arquivo.write(cmd + "\n")
        if texto_complementar:
            for i in range(len(texto_complementar)):
                texto_complementar[i] = texto_complementar[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_complementar[i]}', 'texto_complementar');"
                print(cmd)
                arquivo.write(cmd + "\n")
        if texto_sobrestamento:
            for i in range(len(texto_sobrestamento)):
                texto_sobrestamento[i] = texto_sobrestamento[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestamento[i]}', 'texto_sobrestamento');"
                print(cmd)
                arquivo.write(cmd + "\n")
        if texto_protocolo:
            for i in range(len(texto_protocolo)):
                texto_protocolo[i] = texto_protocolo[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_protocolo[i]}', 'texto_protocolo');"
                print(cmd)
                arquivo.write(cmd + "\n")
            texto_protocolo = []
        if texto_cessionario:
            for i in range(len(texto_cessionario)):
                texto_cessionario[i] = texto_cessionario[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cessionario[i]}', 'cessionarios');"
                print(cmd)
                arquivo.write(cmd + "\n")
            texto_cessionario =[]
        if texto_cedente:
            for i in range(len(texto_cedente)):
                texto_cedente[i] = texto_cedente[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cedente[i]}', 'cedentes');"
                print(cmd)
                arquivo.write(cmd + "\n")
            texto_cedente = []
        #if texto_titulares:
        #    for i in range(len(texto_titulares)):
        #        texto_titulares[i] = texto_titulares[i].replace("'", " ").rstrip('\\')
        #        cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_titulares[i]}', 'titular');"
        #        print(cmd)
        #        arquivo.write(cmd + "\n")

        if texto_sobrestador:
            for i in range(len(texto_sobrestador)):
                texto_sobrestador[i] = texto_sobrestador[i].replace("'", " ").rstrip('\\')
                cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestador[i]}', 'titular');"
                print(cmd)
                arquivo.write(cmd + "\n")
            texto_sobrestador = []
        if codigo_ipas == '009' or codigo_ipas == '576':
            titulares=', '.join(texto_titulares)
            str_deposito = f"'{data_deposito}'"
            if data_deposito is None:
                str_deposito = "NULL"
            str_concessao = f"'{data_concessao}'"
            if data_concessao is None:
                str_concessao = "NULL"
            str_vigencia = f"'{data_vigencia}'"
            if data_vigencia is None:
                str_vigencia = "NULL"
            #titulares = titulares.replace("'", "\\'")
            marca_nome = marca_nome.replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO publicados_ipas_old (numero, titular, data_deposito, data_concessao, data_vigencia, data_out, despacho_out, apresentacao, natureza, nome) VALUES ('{numero}', '{titulares}', {str_deposito}, {str_concessao}, {str_vigencia}, NULL, '', '{marca_apresentacao}', '{marca_natureza}', '{marca_nome}');"
            print(cmd)
            arquivo.write(cmd + "\n")
        if classificacao_vienna_codigo:
            for i in range(len(classificacao_vienna_codigo)):
                cmd = f"INSERT IGNORE INTO classes_vienna_old (id, numero, codigo, edicao, data) VALUES (NULL, '{numero}', '{classificacao_vienna_codigo[i]}', {classificacao_vienna_edicao[i]}, '{data}');"
                print(cmd)
                #arquivo.write(cmd + "\n")
            classificacao_vienna_codigo = []
        if classificacao_nice_codigo:
            for i in range(len(classificacao_nice_codigo)):
                if i >= len(classificacao_nice_codigo):
                    str_codigo = "NULL"
                else:
                    str_codigo = f"'{classificacao_nice_codigo[i]}'"
                if i >= len(classificacao_nice_especificacao):
                    str_especificacao = "NULL"
                else:
                    classificacao_nice_especificacao[i] = classificacao_nice_especificacao[i].replace("'", " ").rstrip('\\')
                    str_especificacao = f"'{classificacao_nice_especificacao[i]}'"
                if i >= len(classificacao_nice_traducao):
                    str_traducao = "NULL"
                else:
                    str_traducao = f"'{classificacao_nice_traducao[i]}'"
                if i >= len(classificacao_nice_status):
                    str_status = "NULL"
                else:
                    str_status = f"'{classificacao_nice_status[i]}'"
                cmd = f"INSERT IGNORE INTO classes_nice_old (id, numero, codigo, especificacao, traducao, status, data) VALUES (NULL, '{numero}', {str_codigo}, {str_especificacao}, {str_traducao}, {str_status}, '{data}');"
                print(cmd)
                #arquivo.write(cmd + "\n")
            classificacao_nice_codigo = []

        if classificacao_nacional_classe:
            for i in range(len(classificacao_nacional_classe)):
                if i >= len(classificacao_nacional_classe):
                    str_classe = "NULL"
                else:
                    str_classe = f"'{classificacao_nacional_classe[i]}'"
                if i >= len(classificacao_nacional_especificacao):
                    str_especificacao = "NULL"
                else:
                    classificacao_nacional_especificacao[i] = classificacao_nacional_especificacao[i].replace("'", " ").rstrip('\\')
                    str_especificacao = f"'{classificacao_nacional_especificacao[i]}'"

                if i >= len(classificacao_nacional_subclasse):
                    str_subclasse = "NULL"
                else:
                    str_subclasse = f"'{classificacao_nacional_subclasse[i]}'"
                cmd = f"INSERT IGNORE INTO classes_nacional_old (id, numero, codigo_classe, especificacao, codigo_subclasse, data) VALUES (NULL, '{numero}', {str_classe}, {str_especificacao}, {str_subclasse}, '{data}');"
                print(cmd)
                #arquivo.write(cmd + "\n")
            classificacao_nacional_classe = []

        if texto_prioridade:
            prioridades=', '.join(texto_prioridade)
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{prioridades}', 'prioridade');"
            print(cmd)
            texto_prioridade = ''
            arquivo.write(cmd + "\n")
        if apostila != '':
            apostila = apostila.replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{apostila}', 'apostila');"
            print(cmd)
            arquivo.write(cmd + "\n")
            apostila = ''
        if procurador != '':
            procurador = procurador.replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{procurador}', 'procurador');"
            print(cmd)
            arquivo.write(cmd + "\n")
            procurador = ''

    despachos_autocontidos = re.findall(r'<despacho codigo="IPAS(\d+)" nome="([^"]+)"\s*/>', line)
    for codigo_ipas, nome_ipas in despachos_autocontidos:
        print(f"Despacho Código: IPAS{codigo_ipas}")
        print(f"Nome: {nome_ipas}")
        data = data_revista
        cmd = f"INSERT IGNORE INTO arquivados_ipas_old (id, codigo_ipas, numero, data, anulado, prmexame) VALUES (NULL, '{codigo_ipas}', '{numero}', '{data}', 0, 0);"
        print(cmd)
        arquivo.write(cmd + "\n")

    if procurador != '':
        procurador = procurador.replace("'", " ").rstrip('\\')
        cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{procurador}', 'procurador');"
        print(cmd)
        arquivo.write(cmd + "\n")
        procurador = ''

    if numero_internacional != '':
        cmd = f"INSERT IGNORE INTO madri_old (numero, numero_internacional, data, data_recebimento) VALUES ('{numero}', '{numero_internacional}', '{data}', '{data_recebimento}');"
        print(cmd)
        numero_internacional = ''
        # arquivo.write(cmd + "\n")
    if nome_ipas != '':
        cmd = f"INSERT IGNORE INTO ipas_old (codigo_ipas, nome_ipas) VALUES ('{codigo_ipas}', '{nome_ipas}');"
        print(cmd)
        nome_ipas = ''
        #arquivo.write(cmd + "\n")
    if texto_complementar:
        for i in range(len(texto_complementar)):
            texto_complementar[i] = texto_complementar[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_complementar[i]}', 'texto_complementar');"
            print(cmd)
            arquivo.write(cmd + "\n")
    if texto_sobrestamento:
        for i in range(len(texto_sobrestamento)):
            texto_sobrestamento[i] = texto_sobrestamento[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestamento[i]}', 'texto_sobrestamento');"
            print(cmd)
            arquivo.write(cmd + "\n")
    if texto_protocolo:
        for i in range(len(texto_protocolo)):
            texto_protocolo[i] = texto_protocolo[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_protocolo[i]}', 'texto_protocolo');"
            print(cmd)
            arquivo.write(cmd + "\n")
        texto_protocolo = []
    if texto_cessionario:
        for i in range(len(texto_cessionario)):
            texto_cessionario[i] = texto_cessionario[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cessionario[i]}', 'cessionarios');"
            print(cmd)
            arquivo.write(cmd + "\n")
        texto_cessionario =[]
    if texto_cedente:
        for i in range(len(texto_cedente)):
            texto_cedente[i] = texto_cedente[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_cedente[i]}', 'cedentes');"
            print(cmd)
            arquivo.write(cmd + "\n")
        texto_cedente = []
    if texto_titulares:
        for i in range(len(texto_titulares)):
            texto_titulares[i] = texto_titulares[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_titulares[i]}', 'titular');"
            print(cmd)
            arquivo.write(cmd + "\n")

    if texto_sobrestador:
        for i in range(len(texto_sobrestador)):
            texto_sobrestador[i] = texto_sobrestador[i].replace("'", " ").rstrip('\\')
            cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{texto_sobrestador[i]}', 'titular');"
            print(cmd)
            arquivo.write(cmd + "\n")
        texto_sobrestador = []
    if codigo_ipas == '009' or codigo_ipas == '576':
        titulares=', '.join(texto_titulares)
        str_deposito = f"'{data_deposito}'"
        if data_deposito is None:
            str_deposito = "NULL"
        str_concessao = f"'{data_concessao}'"
        if data_concessao is None:
            str_concessao = "NULL"
        str_vigencia = f"'{data_vigencia}'"
        if data_vigencia is None:
            str_vigencia = "NULL"
        #titulares = titulares.replace("'", "\\'")
        marca_nome = marca_nome.replace("'", " ").rstrip('\\')
        cmd = f"INSERT IGNORE INTO publicados_ipas_old (numero, titular, data_deposito, data_concessao, data_vigencia, data_out, despacho_out, apresentacao, natureza, nome) VALUES ('{numero}', '{titulares}', {str_deposito}, {str_concessao}, {str_vigencia}, NULL, '', '{marca_apresentacao}', '{marca_natureza}', '{marca_nome}');"
        print(cmd)
        arquivo.write(cmd + "\n")
    if classificacao_vienna_codigo:
        for i in range(len(classificacao_vienna_codigo)):
            cmd = f"INSERT IGNORE INTO classes_vienna_old (id, numero, codigo, edicao, data) VALUES (NULL, '{numero}', '{classificacao_vienna_codigo[i]}', {classificacao_vienna_edicao[i]}, '{data}');"
            print(cmd)
            #arquivo.write(cmd + "\n")
        classificacao_vienna_codigo = []
    if classificacao_nice_codigo:
        for i in range(len(classificacao_nice_codigo)):
            if i >= len(classificacao_nice_codigo):
                str_codigo = "NULL"
            else:
                str_codigo = f"'{classificacao_nice_codigo[i]}'"
            if i >= len(classificacao_nice_especificacao):
                str_especificacao = "NULL"
            else:
                classificacao_nice_especificacao[i] = classificacao_nice_especificacao[i].replace("'", " ").rstrip('\\')
                str_especificacao = f"'{classificacao_nice_especificacao[i]}'"
            if i >= len(classificacao_nice_traducao):
                str_traducao = "NULL"
            else:
                str_traducao = f"'{classificacao_nice_traducao[i]}'"
            if i >= len(classificacao_nice_status):
                str_status = "NULL"
            else:
                str_status = f"'{classificacao_nice_status[i]}'"
            cmd = f"INSERT IGNORE INTO classes_nice_old (id, numero, codigo, especificacao, traducao, status, data) VALUES (NULL, '{numero}', {str_codigo}, {str_especificacao}, {str_traducao}, {str_status}, '{data}');"
            print(cmd)
            #arquivo.write(cmd + "\n")
        classificacao_nice_codigo = []

    if classificacao_nacional_classe:
        for i in range(len(classificacao_nacional_classe)):
            if i >= len(classificacao_nacional_classe):
                str_classe = "NULL"
            else:
                str_classe = f"'{classificacao_nacional_classe[i]}'"
            if i >= len(classificacao_nacional_especificacao):
                str_especificacao = "NULL"
            else:
                classificacao_nacional_especificacao[i] = classificacao_nacional_especificacao[i].replace("'", " ").rstrip('\\')
                str_especificacao = f"'{classificacao_nacional_especificacao[i]}'"

            if i >= len(classificacao_nacional_subclasse):
                str_subclasse = "NULL"
            else:
                str_subclasse = f"'{classificacao_nacional_subclasse[i]}'"
            cmd = f"INSERT IGNORE INTO classes_nacional_old (id, numero, codigo_classe, especificacao, codigo_subclasse, data) VALUES (NULL, '{numero}', {str_classe}, {str_especificacao}, {str_subclasse}, '{data}');"
            print(cmd)
            #arquivo.write(cmd + "\n")
        classificacao_nacional_classe = []

    if texto_prioridade:
        prioridades=', '.join(texto_prioridade)
        cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{prioridades}', 'prioridade');"
        print(cmd)
        texto_prioridade = ''
        arquivo.write(cmd + "\n")
    if apostila != '':
        apostila = apostila.replace("'", " ").rstrip('\\')
        cmd = f"INSERT IGNORE INTO revistas4ipas_old (id, numero, data, codigo_ipas, descricao, tag) VALUES (NULL, '{numero}', '{data}', '{codigo_ipas}', '{apostila}', 'apostila');"
        print(cmd)
        arquivo.write(cmd + "\n")
        apostila = ''

arquivo.close()


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)

